# Fullskip mel unet large data training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook trains the retained full-skip mel-domain U-Net.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


# Large-data mel inpainting training, full-skip 2D U-Net 
This is the base-model training stage. It does not do residual mining. It trains a plain 2D U-Net directly on supervised mel inpainting:

\[
M_{\mathrm{interp}}, m \mapsto \hat M = M_{\mathrm{interp}} + m \odot G(M_{\mathrm{interp}},m).
\]

After this base model is trained, the next stage is residual mining on top of this checkpoint.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip -q install soundfile librosa

In [ ]:
import os, json, time, random
from pathlib import Path
from dataclasses import dataclass
from itertools import cycle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf

print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


 ## Configuration

In [ ]:

SEED = 20260522
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/master")

# ---------------------------------------------------------------------
# Large-data source locations.
# Expected Drive layout:
#   /content/drive/MyDrive/master/Interspeech2025UrgentChallenge/ears.zip
#   /content/drive/MyDrive/master/Interspeech2025UrgentChallenge/libritts.zip
#   /content/drive/MyDrive/master/Interspeech2025UrgentChallenge/vctk.zip
#   /content/drive/MyDrive/master/musdb18hq.zip
# ---------------------------------------------------------------------
RAW_SPEECH_ZIP_DIR = DRIVE_ROOT / "Interspeech2025UrgentChallenge"
SPEECH_ZIP_PATHS = [
    RAW_SPEECH_ZIP_DIR / "ears.zip",
    RAW_SPEECH_ZIP_DIR / "libritts.zip",
    RAW_SPEECH_ZIP_DIR / "vctk.zip",
]
MUSDB_ZIP_PATH = DRIVE_ROOT / "musdb18hq.zip"

# Extract once to Drive. Do not unzip every training run.
DATA_EXTRACT_ROOT = DRIVE_ROOT / "datasets_extracted_large_full"
SPEECH_EXTRACT_ROOT = DATA_EXTRACT_ROOT / "speech"
MUSIC_EXTRACT_ROOT = DATA_EXTRACT_ROOT / "music"

# Persistent split files and documentation summaries.
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"

# First run: True/True/True. Later training reruns: set RUN_DATA_PREP=False.
RUN_DATA_PREP = False
EXTRACT_ZIPS = True
RECREATE_SPLITS = True

TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10
MIN_DURATION_S = 1.50
SPLIT_SEED = 20260522

# Use None for full dataset. For debugging, set e.g. 200.
MAX_SPEECH_FILES = None
MAX_MUSIC_FILES = None

# Choose mode:
#  - "A1" = masked reconstruction only, recommended first
#  - "A2" = mel PatchGAN adversarial training
#  - "A2MS" = multi-scale mel PatchGAN adversarial training
MODE = "A1"
# MODE = "A2"
# MODE = "A2MS"

# Speech/music sampling ratio during training. Keep this close to the intended final mixture.
P_SPEECH = 0.70

# Inpainting k choices. Including k=3 makes training a little closer to harder inserted-frame cases.
K_CHOICES = (1, 2, 3)

# Training. Increase STEPS when additional compute is available.
STEPS      = 100000
LOG_EVERY  = 100
CKPT_EVERY = 1000
SEG_S      = 3.0
BATCH      = 8
NUM_WORKERS = 2   # Set to 0 if multiprocessing causes data loading issues.
PIN_MEMORY = (device.type == "cuda")

# Loss weights. A1 uses LAMBDA_MEL and LAMBDA_DELTA. GAN weights are only used for A2/A2MS.
LAMBDA_MEL = 45.0
LAMBDA_ADV = 1.0
LAMBDA_FM  = 2.0
LAMBDA_DELTA = 0.05

# Warmup for adversarial modes only.
LAMBDA_ADV_MAX = 0.3
ADV_WARMUP_STEPS = 20000
D_UPDATE_EVERY = 2

# Optional identity loss when k=0, unused unless K_CHOICES includes 0.
USE_IDENTITY_LOSS_FOR_K0 = True
LAMBDA_ID = 0.1

# Optional frozen-vocoder STFT loss. Keep False for first large-data base run.
USE_STFT_LOSS = False
LAMBDA_STFT = 0.05
STFT_LOSS_FFTS = (512, 1024, 2048)
STFT_LOSS_HOPS = (128, 256, 512)
STFT_LOSS_WINS = (512, 1024, 2048)
STFT_LOSS_LOGMAG = True

# Optional local staging. For this large dataset, default is False because copying everything to /content is expensive.
STAGE_AUDIO_TO_LOCAL = False
LOCAL_DATA_ROOT = Path("/content/audio_70speech_30music_v1_cache")
LOCAL_SPLIT_DIR = Path("/content/audio_70speech_30music_v1_splits_local")

# Unique run folder.
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
RUN_NAME = "fullskip_mel_unet_large_data"
CKPT_DIR = DRIVE_ROOT / "checkpoints_runs" / f"{RUN_NAME}_{RUN_TAG}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV = CKPT_DIR / "train_log.csv"

print("MODE:", MODE)
print("RUN_NAME:", RUN_NAME)
print("CKPT_DIR:", CKPT_DIR)
print("LOG_CSV:", LOG_CSV)
print("SPLIT_DIR:", SPLIT_DIR)
print("DATA_EXTRACT_ROOT:", DATA_EXTRACT_ROOT)


In [ ]:
# =========================
# One-time data preparation for large-data 2D U-Net base training
# =========================
import csv, zipfile, subprocess
from collections import defaultdict

AUDIO_EXTS = {".wav", ".flac", ".ogg", ".aiff", ".aif"}
MUSDB_MIXTURE_NAMES = {"mixture.wav", "mixture.flac", "mixture.ogg"}

def run(cmd):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True)

def extract_zip_once(zip_path: Path, dest_dir: Path):
    dest_dir.mkdir(parents=True, exist_ok=True)
    marker = dest_dir / ".extract_complete"
    if marker.exists():
        print("Already extracted:", dest_dir)
        return
    assert zip_path.exists(), f"Missing zip: {zip_path}"
    print("Extracting", zip_path, "to", dest_dir)
    # -n = never overwrite existing files. This lets interrupted extraction resume reasonably.
    run(["unzip", "-q", "-n", zip_path, "-d", dest_dir])
    marker.write_text(f"extracted from {zip_path}\n", encoding="utf-8")
    print("Done:", dest_dir)

def safe_duration(path: Path):
    try:
        info = sf.info(str(path))
        if info.frames <= 0 or info.samplerate <= 0:
            return None
        return float(info.frames) / float(info.samplerate)
    except Exception:
        return None

def scan_speech_files(root: Path):
    rows = []
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in AUDIO_EXTS:
            continue
        # dataset label from first directory below SPEECH_EXTRACT_ROOT when possible
        try:
            rel = p.relative_to(SPEECH_EXTRACT_ROOT)
            dataset = rel.parts[0] if len(rel.parts) else "speech"
        except Exception:
            dataset = "speech"
        dur = safe_duration(p)
        if dur is None or dur < MIN_DURATION_S:
            continue
        rows.append(dict(path=str(p), source_type="speech", dataset=dataset, duration_s=dur))
    return rows

def scan_musdb_mixtures(root: Path):
    rows = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.name.lower() not in MUSDB_MIXTURE_NAMES:
            continue
        dur = safe_duration(p)
        if dur is None or dur < MIN_DURATION_S:
            continue
        # Track name is usually parent directory of mixture.wav.
        track = p.parent.name
        rows.append(dict(path=str(p), source_type="music", dataset="musdb18hq_mixture", track=track, duration_s=dur))
    return rows

def write_list(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(r["path"] for r in rows) + ("\n" if rows else ""), encoding="utf-8")

def split_rows(rows, train_frac=0.8, val_frac=0.1, seed=0, max_files=None):
    rows = list(rows)
    rng = random.Random(seed)
    rng.shuffle(rows)
    if max_files is not None:
        rows = rows[:int(max_files)]
    n = len(rows)
    n_train = int(round(train_frac * n))
    n_val = int(round(val_frac * n))
    train = rows[:n_train]
    val = rows[n_train:n_train+n_val]
    test = rows[n_train+n_val:]
    return train, val, test

def summarize_rows(rows, label):
    by = defaultdict(float)
    for r in rows:
        by[(r.get("source_type", "?"), r.get("dataset", "?"))] += float(r.get("duration_s", 0.0))
    print("\n", label)
    total = sum(by.values())
    print(f"total: {total/3600:.2f} h, files: {len(rows)}")
    for (source, dataset), sec in sorted(by.items()):
        print(f"  {source:6s} {dataset:30s}: {sec/3600:8.2f} h")

def write_manifest_csv(path: Path, rows):
    if not rows:
        return
    keys = sorted(set().union(*[set(r.keys()) for r in rows]))
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

if RUN_DATA_PREP:
    if EXTRACT_ZIPS:
        for zp in SPEECH_ZIP_PATHS:
            extract_zip_once(Path(zp), SPEECH_EXTRACT_ROOT / Path(zp).stem)
        extract_zip_once(MUSDB_ZIP_PATH, MUSIC_EXTRACT_ROOT / MUSDB_ZIP_PATH.stem)

    print("Scanning extracted speech files...")
    speech_rows = scan_speech_files(SPEECH_EXTRACT_ROOT)
    print("Scanning MUSDB18-HQ mixture files only...")
    music_rows = scan_musdb_mixtures(MUSIC_EXTRACT_ROOT)

    assert len(speech_rows) > 0, f"No speech audio files found under {SPEECH_EXTRACT_ROOT}"
    assert len(music_rows) > 0, f"No MUSDB mixture.wav files found under {MUSIC_EXTRACT_ROOT}. Check extraction path."

    summarize_rows(speech_rows, "All speech data")
    summarize_rows(music_rows, "All music mixture data")

    if RECREATE_SPLITS:
        sp_train, sp_val, sp_test = split_rows(speech_rows, TRAIN_FRAC, VAL_FRAC, SPLIT_SEED, MAX_SPEECH_FILES)
        mu_train, mu_val, mu_test = split_rows(music_rows, TRAIN_FRAC, VAL_FRAC, SPLIT_SEED + 1, MAX_MUSIC_FILES)

        write_list(SPLIT_DIR / "speech_train.txt", sp_train)
        write_list(SPLIT_DIR / "speech_val.txt", sp_val)
        write_list(SPLIT_DIR / "speech_test.txt", sp_test)
        write_list(SPLIT_DIR / "music_train.txt", mu_train)
        write_list(SPLIT_DIR / "music_val.txt", mu_val)
        write_list(SPLIT_DIR / "music_test.txt", mu_test)

        all_rows = []
        for split_name, rows in [
            ("speech_train", sp_train), ("speech_val", sp_val), ("speech_test", sp_test),
            ("music_train", mu_train), ("music_val", mu_val), ("music_test", mu_test),
        ]:
            for r in rows:
                rr = dict(r)
                rr["split"] = split_name
                all_rows.append(rr)
        write_manifest_csv(SPLIT_DIR / "manifest_with_durations.csv", all_rows)

        print("\nWrote splits to:", SPLIT_DIR)
        for name, rows in [
            ("speech_train", sp_train), ("speech_val", sp_val), ("speech_test", sp_test),
            ("music_train", mu_train), ("music_val", mu_val), ("music_test", mu_test),
        ]:
            hrs = sum(float(r["duration_s"]) for r in rows) / 3600
            print(f"{name:13s}: {len(rows):6d} files, {hrs:8.2f} h")
else:
    print("RUN_DATA_PREP=False. Reusing existing extracted data and split files:", SPLIT_DIR)

# Always print split-hour summary from current split files for documentation.
def split_hours(split_file):
    p = SPLIT_DIR / split_file
    if not p.exists():
        return 0, 0.0
    paths = [Path(x.strip()) for x in p.read_text().splitlines() if x.strip()]
    total = 0.0
    ok = 0
    for path in paths:
        dur = safe_duration(path)
        if dur is not None:
            total += dur
            ok += 1
    return ok, total / 3600

print("\nCurrent split duration summary:")
summary_rows = []
for sf in ["speech_train.txt", "speech_val.txt", "speech_test.txt", "music_train.txt", "music_val.txt", "music_test.txt"]:
    n, h = split_hours(sf)
    print(f"{sf:18s}: {n:6d} files, {h:8.2f} h")
    summary_rows.append({"split_file": sf, "files_with_duration": n, "hours": h})
pd.DataFrame(summary_rows).to_csv(SPLIT_DIR / "split_hours_summary.csv", index=False)
print("Saved:", SPLIT_DIR / "split_hours_summary.csv")


In [ ]:
RUN_CONFIG = dict(
    mode=MODE,
    seed=SEED,
    split_dir=str(SPLIT_DIR),
    prepared_dataset_dir=str(PREPARED_DATASET_DIR),
    data_extract_root=str(DATA_EXTRACT_ROOT),
    speech_zip_paths=[str(p) for p in SPEECH_ZIP_PATHS],
    musdb_zip_path=str(MUSDB_ZIP_PATH),
    p_speech=P_SPEECH,
    k_choices=list(K_CHOICES),
    steps=STEPS,
    log_every=LOG_EVERY,
    ckpt_every=CKPT_EVERY,
    seg_s=SEG_S,
    batch=BATCH,
    num_workers=NUM_WORKERS,
    lambda_mel=LAMBDA_MEL,
    lambda_adv=LAMBDA_ADV,
    lambda_fm=LAMBDA_FM,
    adv_warmup_steps=ADV_WARMUP_STEPS,
    use_identity_loss_for_k0=USE_IDENTITY_LOSS_FOR_K0,
    lambda_id=LAMBDA_ID,
    use_stft_loss=USE_STFT_LOSS,
    lambda_stft=LAMBDA_STFT,
    G_arch="FullSkipUNet2D",
    stage_audio_to_local=STAGE_AUDIO_TO_LOCAL,
)

(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print(json.dumps(RUN_CONFIG, indent=2))
print("Saved run config:", CKPT_DIR / "run_config.json")


 ## HiFi-GAN mel_spectrogram config (for consistent mel params) We only need HiFi-GAN config + meldataset.mel_spectrogram.

In [ ]:
HIFIGAN_DRIVE_DIR = DRIVE_ROOT / "hifi-gan"
assert HIFIGAN_DRIVE_DIR.exists(), f"Missing HiFi-GAN repo at: {HIFIGAN_DRIVE_DIR}"

os.system("rm -rf /content/hifi-gan")
os.system(f"ln -s '{HIFIGAN_DRIVE_DIR}' /content/hifi-gan")

import sys
HIFIGAN_DIR = Path("/content/hifi-gan")
if str(HIFIGAN_DIR) not in sys.path:
    sys.path.insert(0, str(HIFIGAN_DIR))

from env import AttrDict
from meldataset import mel_spectrogram

# Patch librosa mel fn keyword issues (prevents TypeError in some setups)
import librosa.filters as LF
def _patched_librosa_mel_fn(sr, n_fft, n_mels, fmin, fmax):
    return LF.mel(sr=sr, n_fft=n_fft, n_mels=n_mels, fmin=fmin, fmax=fmax)

if "meldataset" in sys.modules:
    if getattr(sys.modules["meldataset"], "librosa_mel_fn", None) is not _patched_librosa_mel_fn:
        sys.modules["meldataset"].librosa_mel_fn = _patched_librosa_mel_fn
        print("Patched meldataset.librosa_mel_fn")

UNIV_DIR = HIFIGAN_DIR / "pretrained" / "UNIVERSAL_V1"
cfg_path = UNIV_DIR / "config.json"
if not cfg_path.exists():
    cfg_path = UNIV_DIR / "config"
h = AttrDict(json.loads(cfg_path.read_text()))

SR = int(getattr(h, "sampling_rate", getattr(h, "sample_rate", 22050)))
N_MELS = int(h.num_mels)

print("SR:", SR, "N_MELS:", N_MELS)
print("mel params:", dict(n_fft=h.n_fft, hop_size=h.hop_size, win_size=h.win_size, fmin=h.fmin, fmax=h.fmax))

In [ ]:
from models import Generator

g_candidates = sorted([p for p in UNIV_DIR.glob("g_*") if p.is_file()])
assert g_candidates, f"No g_* found in {UNIV_DIR}"
g_ckpt = g_candidates[-1]
print("Loading HiFi-GAN generator:", g_ckpt.name)

hifigan_G = Generator(h).to(device).eval()
ckpt_obj = torch.load(str(g_ckpt), map_location=device)

if isinstance(ckpt_obj, dict) and "generator" in ckpt_obj:
    sd = ckpt_obj["generator"]
elif isinstance(ckpt_obj, dict) and "state_dict" in ckpt_obj:
    sd = ckpt_obj["state_dict"]
else:
    sd = ckpt_obj

missing, unexpected = hifigan_G.load_state_dict(sd, strict=False)
print("HiFi-GAN load missing:", len(missing), "unexpected:", len(unexpected))

if hasattr(hifigan_G, "remove_weight_norm"):
    hifigan_G.remove_weight_norm()

for p in hifigan_G.parameters():
    p.requires_grad_(False)

# Sanity check
with torch.no_grad():
    y_hat = hifigan_G(torch.zeros(1, N_MELS, 10, device=device))
print("HiFi-GAN dummy output shape:", tuple(y_hat.shape))

In [ ]:
# =========================
# Optional local staging.
# =========================
# For the full large dataset, copying all audio to /content can take a long time and may exceed local disk.
# Default: use the prepared split paths directly from SPLIT_DIR.
# For a smaller debug subset, set STAGE_AUDIO_TO_LOCAL=True in the config cell.

import shutil
from pathlib import Path

if STAGE_AUDIO_TO_LOCAL:
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"
    LISTS = ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt", "speech_test.txt", "music_test.txt"]

    def _read_lines(p: Path):
        lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
        return [ln for ln in lines if ln]

    def to_local_path(p: Path) -> Path:
        p = Path(p)
        for root in [DRIVE_ROOT, Path("/content/drive")]:
            try:
                rel = p.relative_to(root)
                return LOCAL_DATA_ROOT / rel
            except Exception:
                pass
        return LOCAL_DATA_ROOT / "flat" / p.name

    all_files = []
    for name in LISTS:
        src_list = SPLIT_DIR / name
        assert src_list.exists(), f"Missing split list: {src_list}"
        all_files += [Path(x) for x in _read_lines(src_list)]

    all_files = sorted(set(all_files))
    print("Unique audio files referenced by train/val splits:", len(all_files))

    copied = 0
    skipped = 0
    for src in all_files:
        dst = to_local_path(src)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1

    print(f"Copied {copied} files, skipped {skipped} existing.")
    original_split_dir = SPLIT_DIR
    for name in LISTS:
        src_paths = [Path(x) for x in _read_lines(original_split_dir / name)]
        dst_paths = [str(to_local_path(p)) for p in src_paths]
        (LOCAL_SPLIT_DIR / name).write_text("\n".join(dst_paths) + "\n", encoding="utf-8")
    SPLIT_DIR = LOCAL_SPLIT_DIR
    RUN_CONFIG["split_dir"] = str(SPLIT_DIR)
    RUN_CONFIG["local_data_root"] = str(LOCAL_DATA_ROOT)
    (CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
    print("Now using local SPLIT_DIR:", SPLIT_DIR)
else:
    print("STAGE_AUDIO_TO_LOCAL=False. Using split files directly from Drive/extracted paths:", SPLIT_DIR)

# Verify split lists exist and show the duration summary that was written during data prep.
for name in ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt"]:
    p = SPLIT_DIR / name
    assert p.exists(), f"Missing list: {p}"
    lines = [ln for ln in p.read_text().splitlines() if ln.strip()]
    print(name, "files:", len(lines), "first:", lines[0] if lines else "EMPTY")

summary_path = SPLIT_DIR / "split_hours_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("No split_hours_summary.csv found yet.")


In [ ]:
# =========================
# Dataset + loaders (speech/music separate, infinite iterators)
# =========================
from torch.utils.data import Dataset, DataLoader

def read_list(p: Path):
    assert p.exists(), f"Missing list file: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav

class FileListDataset(Dataset):
    def __init__(self, list_path: Path, sr: int, segment_s: float):
        self.paths = read_list(list_path)
        self.sr = sr
        self.seg_len = int(sr * segment_s)
        assert len(self.paths) > 0, f"Empty list: {list_path}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        w = safe_load_wav_mono(self.paths[idx], self.sr)
        if w.numel() < self.seg_len:
            w = F.pad(w, (0, self.seg_len - w.numel()))
        max_start = w.numel() - self.seg_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        return w[start:start+self.seg_len]

# Split list paths after optional local staging
speech_train = SPLIT_DIR / "speech_train.txt"
music_train  = SPLIT_DIR / "music_train.txt"
speech_val   = SPLIT_DIR / "speech_val.txt"
music_val    = SPLIT_DIR / "music_val.txt"

ds_speech = FileListDataset(speech_train, SR, SEG_S)
ds_music  = FileListDataset(music_train,  SR, SEG_S)


dl_speech = DataLoader(ds_speech, batch_size=BATCH, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
                       persistent_workers=(NUM_WORKERS > 0))
dl_music  = DataLoader(ds_music,  batch_size=BATCH, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
                       persistent_workers=(NUM_WORKERS > 0))

def infinite(dl):
    while True:
        for b in dl:
            yield b

it_speech = infinite(dl_speech)
it_music  = infinite(dl_music)

def next_mixed_batch():
    b = next(it_speech) if random.random() < P_SPEECH else next(it_music)
    return b.to(device)

print("speech files:", len(ds_speech), "music files:", len(ds_music), "P_SPEECH:", P_SPEECH)

# Verify that batches are loaded from local /content paths.
print("Example speech path:", read_list(speech_train)[0])

In [ ]:
## Inpainting corruption + masked L1 (anchors vs missing frames)

In [ ]:
def linear_time_inpaint(mel: torch.Tensor, k: int, offset: int):
    """
    mel: (B, n_mels, T)
    k: number of missing frames between anchors
       - k=1 => pattern: anchor, missing, anchor, missing, ...
       - k=2 => anchor, missing, missing, anchor, ...
       - k=0 => no missing frames (identity case)
    offset: shift pattern in [0, k]
    """
    B, C, T = mel.shape
    step = k + 1

    mel_interp = mel.clone()
    mask = torch.zeros((B, 1, T), device=mel.device, dtype=mel.dtype)

    if k == 0:
        return mel_interp, mask

    for left in range(offset, T - step, step):
        right = left + step
        # we fill intermediate frames by linear interpolation
        for j in range(1, step):
            t = left + j
            alpha = j / step
            mel_interp[:, :, t] = (1.0 - alpha) * mel[:, :, left] + alpha * mel[:, :, right]
            mask[:, :, t] = 1.0

    return mel_interp, mask

def make_inpainting_pair(mel_real: torch.Tensor, k_choices=(1,2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0
    mel_interp, mask = linear_time_inpaint(mel_real, k=k, offset=offset)
    return {"real": mel_real, "interp": mel_interp, "mask": mask, "k": k, "offset": offset}

def masked_l1(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, w_missing=1.0, w_anchor=0.0):
    """
    Applies L1 only where mask==1 (missing frames) by default.
    """
    m = mask.expand(-1, pred.shape[1], -1)  # (B, n_mels, T)
    w = w_anchor + (w_missing - w_anchor) * m
    err = (pred - target).abs()
    return (w * err).sum() / (w.sum() + 1e-8)

 ## Generator G (MelRefiner): predicts delta; only applied on missing frames

In [ ]:
# class MelRefiner(nn.Module):
#     def __init__(self, n_mels: int, hidden: int = 128, n_blocks: int = 4, use_mask: bool = True):
#         super().__init__()
#         self.use_mask = use_mask
#         c_in = n_mels + (1 if use_mask else 0)
#         self.in_conv = nn.Conv1d(c_in, hidden, kernel_size=5, padding=2)

#         blocks = []
#         for _ in range(n_blocks):
#             blocks += [
#                 nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
#                 nn.LeakyReLU(0.2, inplace=True),
#                 nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
#                 nn.LeakyReLU(0.2, inplace=True),
#             ]
#         self.blocks = nn.Sequential(*blocks)

#         self.out_conv = nn.Conv1d(hidden, n_mels, kernel_size=3, padding=1)
#         nn.init.zeros_(self.out_conv.weight)
#         nn.init.zeros_(self.out_conv.bias)

#     def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
#         x = torch.cat([mel_interp, mask], dim=1) if self.use_mask else mel_interp
#         h = F.leaky_relu(self.in_conv(x), 0.2)
#         h = self.blocks(h)
#         delta = self.out_conv(h)
#         return delta

# G = MelRefiner(N_MELS, hidden=128, n_blocks=4, use_mask=True).to(device)
# print("G params (M):", sum(p.numel() for p in G.parameters())/1e6)

In [ ]:
# =========================
# 2D U-Net generator for mel inpainting
# Full-skip 2D U-Net generator for mel inpainting
# =========================

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8):
        super().__init__()
        g = min(groups, out_ch)
        while out_ch % g != 0 and g > 1:
            g -= 1
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, k=3, s=1, p=1, groups=groups),
            ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, groups=groups),
        )

    def forward(self, x):
        return self.net(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2, 2), groups=8):
        super().__init__()
        g = min(groups, out_ch)
        while out_ch % g != 0 and g > 1:
            g -= 1

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, groups=groups),
        )

    def forward(self, x):
        return self.net(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups)

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)


class MelRefinerUNet2D(nn.Module):
    """
    Full-skip 2D U-Net over (frequency x time).

    Inputs:
      mel_interp: (B, n_mels, T)
      mask:       (B, 1, T)

    Output:
      delta:      (B, n_mels, T)

    Notes:
      - Uses the mask as an extra 2D input channel.
      - Final conv is zero-initialized so training starts near identity/residual=0.
      - Uses a standard full-resolution U-Net fusion stage: the final decoder
        features are concatenated with Enc1 features and passed through DoubleConv.
      - Non-causal in time (uses context on both sides), which is fine for offline inpainting.
    """
    def __init__(self, n_mels: int, base: int = 24, use_mask: bool = True, groups: int = 8):
        super().__init__()
        self.n_mels = n_mels
        self.use_mask = use_mask

        c_in = 1 + (1 if use_mask else 0)  # mel + optional mask channel

        # Encoder
        self.enc1 = DoubleConv(c_in,       base,     groups=groups)        # full res
        self.enc2 = DownBlock(base,        base*2,   stride=(2, 2), groups=groups)
        self.enc3 = DownBlock(base*2,      base*4,   stride=(2, 2), groups=groups)
        self.enc4 = DownBlock(base*4,      base*4,   stride=(1, 2), groups=groups)
        self.bot  = DoubleConv(base*4,     base*4,   groups=groups)

        # Decoder
        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)  # bot + enc4
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)  # up3 + enc3
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)  # up2 + enc2

        # Full-resolution fusion and output head.
        # This replaces the compact additive final skip with the more standard
        # U-Net pattern: upsample -> concat Enc1 -> DoubleConv -> output conv.
        self.final_fuse = DoubleConv(base + base, base, groups=groups)
        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)

        # Start from "do almost nothing" residual prediction
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        # mel_interp: (B, F, T) -> (B, 1, F, T)
        x = mel_interp.unsqueeze(1)

        if self.use_mask:
            # mask: (B, 1, T) -> broadcast to (B, 1, F, T)
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)

        # Encoder
        s1 = self.enc1(x)   # (B, base,   F,   T)
        s2 = self.enc2(s1)  # (B, 2base,  F/2, T/2)
        s3 = self.enc3(s2)  # (B, 4base,  F/4, T/4)
        s4 = self.enc4(s3)  # (B, 4base,  ~F/4, T/8)

        # Bottleneck
        b = self.bot(s4)

        # Decoder
        u3 = self.up3(b,  s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)

        # Back to original resolution
        u0 = F.interpolate(
            u1,
            size=s1.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        # Full-resolution U-Net skip fusion from shallow encoder features.
        # u0 and s1 both have shape (B, base, F, T). Concatenation gives
        # (B, 2*base, F, T), followed by a learnable DoubleConv fusion block.
        u0 = torch.cat([u0, s1], dim=1)
        u0 = self.final_fuse(u0)

        # Output delta on mel plane
        delta = self.out_conv(u0).squeeze(1)   # (B, n_mels, T)
        return delta


# Instantiate generator
# base=24 is a good default; if GPU memory is tight, try base=16.
G = MelRefinerUNet2D(
    n_mels=N_MELS,
    base=24,
    use_mask=True,
    groups=8,
).to(device)

print("G params (M):", sum(p.numel() for p in G.parameters()) / 1e6)

# Quick shape smoke test
with torch.no_grad():
    mel_dummy = torch.zeros(2, N_MELS, 100, device=device)
    mask_dummy = torch.zeros(2, 1, 100, device=device)
    out_dummy = G(mel_dummy, mask_dummy)
print("Dummy output shape:", tuple(out_dummy.shape))

""" ## Discriminator D (A2 / A2MS): Mel PatchGAN with spectral norm + feature maps"""

In [ ]:
def set_requires_grad(model, flag: bool):
    for p in model.parameters():
        p.requires_grad_(flag)

def pack_mel_for_D(mel: torch.Tensor, mask: torch.Tensor):
    """
    mel: (B, n_mels, T)
    mask:(B, 1, T)
    => x: (B, 2, n_mels, T) where channel0=mel, channel1=mask broadcast across mel bins
    """
    x_mel  = mel.unsqueeze(1)  # (B,1,n_mels,T)
    x_mask = mask.unsqueeze(2).expand(-1, 1, mel.shape[1], -1)  # (B,1,n_mels,T)
    return torch.cat([x_mel, x_mask], dim=1)  # (B,2,n_mels,T)

class MelPatchDiscriminator(nn.Module):
    def __init__(self, in_channels: int = 2, base: int = 32):
        super().__init__()
        SN = nn.utils.spectral_norm

        def conv(in_ch, out_ch, k, s, p):
            return SN(nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p))

        self.blocks = nn.ModuleList([
            nn.Sequential(conv(in_channels, base,   (5, 5), (1, 2), (2, 2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(conv(base,        base*2, (5, 5), (2, 2), (2, 2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(conv(base*2,      base*4, (3, 3), (2, 2), (1, 1)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(conv(base*4,      base*8, (3, 3), (2, 2), (1, 1)), nn.LeakyReLU(0.2, inplace=True)),
        ])
        self.out = conv(base*8, 1, (3, 3), (1, 1), (1, 1))

    def forward(self, x):
        fmaps = []
        h = x
        for blk in self.blocks:
            h = blk(h)
            fmaps.append(h)
        return self.out(h), fmaps  # logits map + feature maps

class MultiScaleMelDiscriminator(nn.Module):
    """
    Runs several MelPatchDiscriminators at different time scales.
    Scale is implemented by average pooling along time (T) before feeding D.
    """
    def __init__(self, in_channels=2, base=32, scales=(1, 2, 4)):
        super().__init__()
        self.scales = scales
        self.ds = nn.ModuleList([MelPatchDiscriminator(in_channels=in_channels, base=base) for _ in scales])

    def _downsample_time(self, x, s: int):
        # x: (B, 2, n_mels, T)
        if s == 1:
            return x
        return F.avg_pool2d(x, kernel_size=(1, s), stride=(1, s), ceil_mode=False)

    def forward(self, x):
        logits_list = []
        fmaps_list = []
        for D_i, s in zip(self.ds, self.scales):
            xs = self._downsample_time(x, s)
            logits, fmaps = D_i(xs)
            logits_list.append(logits)
            fmaps_list.append(fmaps)
        return logits_list, fmaps_list

def d_loss_lsgan(d_real, d_fake):
    return ((d_real - 1.0) ** 2).mean() + ((d_fake - 0.0) ** 2).mean()

def g_adv_loss_lsgan(d_fake):
    return ((d_fake - 1.0) ** 2).mean()

def fm_loss(fmaps_real, fmaps_fake):
    return sum((fr - ff).abs().mean() for fr, ff in zip(fmaps_real, fmaps_fake))

def d_loss_lsgan_ms(d_reals, d_fakes):
    loss = 0.0
    for dr, df in zip(d_reals, d_fakes):
        loss = loss + ((dr - 1.0) ** 2).mean() + ((df - 0.0) ** 2).mean()
    return loss / len(d_reals)

def g_adv_loss_lsgan_ms(d_fakes):
    loss = 0.0
    for df in d_fakes:
        loss = loss + ((df - 1.0) ** 2).mean()
    return loss / len(d_fakes)

def fm_loss_ms(fmaps_reals, fmaps_fakes):
    loss = 0.0
    n = 0
    for fr_scale, ff_scale in zip(fmaps_reals, fmaps_fakes):
        for fr, ff in zip(fr_scale, ff_scale):
            loss = loss + (fr - ff).abs().mean()
            n += 1
    return loss / max(1, n)

if MODE == "A2":
    D = MelPatchDiscriminator(in_channels=2, base=32).to(device)
    print("D params (M):", sum(p.numel() for p in D.parameters())/1e6)

if MODE == "A2MS":
    D = MultiScaleMelDiscriminator(in_channels=2, base=32, scales=(1,2,4)).to(device)
    print("MS-D params (M):", sum(p.numel() for p in D.parameters())/1e6)

In [ ]:
def _stft_mag(x: torch.Tensor, n_fft: int, hop: int, win: int, eps: float = 1e-7):
    """
    x: (B, T)
    returns magnitude spectrogram: (B, F, TT)
    """
    window = torch.hann_window(win, device=x.device)
    X = torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=window,
        center=True,
        return_complex=True,
    )
    mag = X.abs().clamp_min(eps)
    return mag

def mrstft_loss(
    x_hat: torch.Tensor,
    x_tgt: torch.Tensor,
    fft_sizes=(512, 1024, 2048),
    hop_sizes=(128, 256, 512),
    win_sizes=(512, 1024, 2048),
    use_logmag=True,
):
    """
    x_hat, x_tgt: (B, T)
    returns scalar tensor
    """
    assert len(fft_sizes) == len(hop_sizes) == len(win_sizes)

    loss = 0.0
    n = 0

    for n_fft, hop, win in zip(fft_sizes, hop_sizes, win_sizes):
        mag_hat = _stft_mag(x_hat, n_fft=n_fft, hop=hop, win=win)
        mag_tgt = _stft_mag(x_tgt, n_fft=n_fft, hop=hop, win=win)

        # spectral convergence-ish L1 magnitude
        l_mag = (mag_hat - mag_tgt).abs().mean()

        if use_logmag:
            l_log = (mag_hat.log() - mag_tgt.log()).abs().mean()
            loss = loss + l_mag + l_log
            n += 2
        else:
            loss = loss + l_mag
            n += 1

    return loss / max(1, n)

In [ ]:
def mel_to_wave_vocoder(mel_bt: torch.Tensor):
    """
    mel_bt: (B, n_mels, T)
    returns waveform: (B, T_wav)
    """
    y = hifigan_G(mel_bt)
    return y.squeeze(1) if y.ndim == 3 and y.shape[1] == 1 else y

 ## Optimizers + checkpoint save/resume

In [ ]:
opt_g = torch.optim.AdamW(G.parameters(), lr=2e-4, betas=(0.9, 0.99), weight_decay=1e-4)

if MODE in ["A2", "A2MS"]:
    # opt_d = torch.optim.AdamW(D.parameters(), lr=2e-4, betas=(0.8, 0.99), weight_decay=1e-4)
    opt_d = torch.optim.AdamW(D.parameters(), lr=1e-4, betas=(0.8, 0.99), weight_decay=1e-4)  # was 2e-4

def save_ckpt(step: int):
    obj = {
        "step": step,
        "run_config": RUN_CONFIG,
        "mode": MODE,
        "seed": SEED,
        "k_choices": list(K_CHOICES),
        "p_speech": P_SPEECH,
        "G": G.state_dict(),
        "opt_g": opt_g.state_dict(),
        "hifigan_config": dict(h),
        "use_stft_loss": USE_STFT_LOSS,
        "lambda_stft": LAMBDA_STFT,
    }
    if MODE in ["A2", "A2MS"]:
        obj.update({"D": D.state_dict(), "opt_d": opt_d.state_dict()})

    p = CKPT_DIR / f"{MODE}_step{step:07d}.pt"
    torch.save(obj, p)
    torch.save(obj, CKPT_DIR / "latest.pt")
    print("saved:", p)


# A2_20260304_112559
# A2_MRSTFT_20260309_215854

# A2_MRSTFT_20260310_060839
RESUME_PATH = None

# RESUME_PATH = Path("/content/drive/MyDrive/master/checkpoints_runs/<run_name>/latest.pt")

# RESUME_PATH = Path("/content/drive/MyDrive/master/checkpoints_runs/<run_name>/latest.pt")

print("RESUME_PATH:", RESUME_PATH)
# print("Exists:", RESUME_PATH.exists())

start_step = 0
if RESUME_PATH is not None and RESUME_PATH.exists():
    ck = torch.load(str(RESUME_PATH), map_location="cpu")
    G.load_state_dict(ck["G"], strict=True)
    opt_g.load_state_dict(ck["opt_g"])

    if MODE in ["A2", "A2MS"]:
        assert "D" in ck and "opt_d" in ck, "Checkpoint missing D/opt_d for A2 resume."
        D.load_state_dict(ck["D"], strict=True)
        opt_d.load_state_dict(ck["opt_d"])

    start_step = int(ck.get("step", 0))
    print("Resumed:", RESUME_PATH)
    print("start_step:", start_step)
else:
    print("Starting fresh.")


if RESUME_PATH is not None and RESUME_PATH.exists():
    print("Resuming from existing checkpoint, but saving into current CKPT_DIR:", CKPT_DIR)
else:
    print("Starting fresh run in:", CKPT_DIR)


# # Write CSV header
# if not LOG_CSV.exists():
#     LOG_CSV.write_text(
#         "step,lossD,lossG,loss_mel,loss_adv,loss_fm,loss_stft,base_mel,ref_mel,imp_pct,k,offset,dr_mean,df_mean,dr_sig,df_sig,minutes\n",
#         encoding="utf-8"
#     )

# --- CSV header ---
if not LOG_CSV.exists():
    LOG_CSV.write_text(
        "step,lossD,lossG,loss_mel,loss_adv,loss_fm,loss_stft,"
        "base_mel,ref_mel,delta_mel,ratio_mel,win_mel,"
        "k,offset,dr_mean,df_mean,dr_sig,df_sig,minutes\n",
        encoding="utf-8"
    )


print("MODE:", MODE)
print("RUN_NAME:", RUN_NAME)
print("CKPT_DIR:", CKPT_DIR)
print("LOG_CSV:", LOG_CSV)



 ## Training loop (prints baseline->refinement improvement, and D score stats)

In [ ]:
# =========================
# Training loop (A1 / A2 / A2MS)
# =========================

# Required configuration variables for the training loop
# LAMBDA_MEL, LAMBDA_FM, LAMBDA_ID, ADV_WARMUP_STEPS
# LAMBDA_ADV_MAX, LAMBDA_DELTA
# D_UPDATE_EVERY, LOG_EVERY, CKPT_EVERY, STEPS, SEG_S
# USE_IDENTITY_LOSS_FOR_K0
# And the helpers: set_requires_grad, pack_mel_for_D, masked_l1, etc.

G.train()
if MODE in ["A2", "A2MS"]:
    D.train()

t0 = time.time()

for step in range(start_step + 1, start_step + STEPS + 1):
    # -------------------------
    # 1) batch -> mel
    # -------------------------
    wav = next_mixed_batch()  # (B, T) already on device

    mel_real = mel_spectrogram(
        wav,
        n_fft=h.n_fft,
        num_mels=h.num_mels,
        sampling_rate=h.sampling_rate,
        hop_size=h.hop_size,
        win_size=h.win_size,
        fmin=h.fmin,
        fmax=h.fmax,
        center=False,
    )  # (B, n_mels, Tm)

    pair = make_inpainting_pair(mel_real, k_choices=K_CHOICES, randomize_offset=True)
    mel_interp, mask = pair["interp"], pair["mask"]

    # baseline error = interpolation vs real (only missing frames)
    with torch.no_grad():
        base_mel = masked_l1(mel_interp, mel_real, mask, 1.0, 0.0)

    # -------------------------
    # 2) A1: recon only
    # -------------------------
    if MODE == "A1":
        delta = G(mel_interp, mask)
        mel_fixed = mel_interp + mask * delta

        missing = mask.sum()
        loss_id = torch.tensor(0.0, device=device)

        if USE_IDENTITY_LOSS_FOR_K0 and (missing < 1.0):
            # encourage delta≈0 when nothing is missing
            loss_mel = torch.tensor(0.0, device=device)
            loss_id  = delta.abs().mean()
        else:
            loss_mel = masked_l1(mel_fixed, mel_real, mask, 1.0, 0.0)

        loss_adv = torch.tensor(0.0, device=device)
        loss_fm  = torch.tensor(0.0, device=device)
        loss_d   = torch.tensor(0.0, device=device)

        # optional delta regularizer
        loss_delta = (mask * delta).abs().mean()

        if USE_STFT_LOSS:
            wav_hat = mel_to_wave_vocoder(mel_fixed)
            with torch.no_grad():
                wav_tgt = mel_to_wave_vocoder(mel_real)

            loss_stft = mrstft_loss(
                wav_hat, wav_tgt,
                fft_sizes=STFT_LOSS_FFTS,
                hop_sizes=STFT_LOSS_HOPS,
                win_sizes=STFT_LOSS_WINS,
                use_logmag=STFT_LOSS_LOGMAG,
            )
        else:
            loss_stft = torch.tensor(0.0, device=device)

        loss_g = (
            (LAMBDA_MEL * loss_mel) +
            (LAMBDA_ID * loss_id) +
            (LAMBDA_DELTA * loss_delta) +
            (LAMBDA_STFT * loss_stft)
        )

        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_g.step()

        dr_mean = df_mean = dr_sig = df_sig = 0.0

    # -------------------------
    # 3) A2 / A2MS: GAN
    # -------------------------
    else:
        # =========================================================
        # 3a) D step (optionally skipped)
        # =========================================================
        do_d_update = (step % D_UPDATE_EVERY) == 0

        if do_d_update:
            set_requires_grad(D, True)

            # Generate fake mel without tracking G gradients
            with torch.no_grad():
                delta_ng = G(mel_interp, mask)
                mel_fixed_ng = mel_interp + mask * delta_ng

            x_real = pack_mel_for_D(mel_real, mask)
            x_fake = pack_mel_for_D(mel_fixed_ng.detach(), mask)

            d_real, f_real = D(x_real)
            d_fake, f_fake = D(x_fake)

            if MODE == "A2":
                loss_d = d_loss_lsgan(d_real, d_fake)
            else:
                loss_d = d_loss_lsgan_ms(d_real, d_fake)

            opt_d.zero_grad(set_to_none=True)
            loss_d.backward()
            torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
            opt_d.step()

        else:
            # No discriminator update this step; still compute scores for logging
            loss_d = torch.tensor(0.0, device=device)
            with torch.no_grad():
                delta_ng = G(mel_interp, mask)
                mel_fixed_ng = mel_interp + mask * delta_ng
                x_real = pack_mel_for_D(mel_real, mask)
                x_fake = pack_mel_for_D(mel_fixed_ng, mask)
                d_real, f_real = D(x_real)
                d_fake, f_fake = D(x_fake)

        # =========================================================
        # 3b) G step
        # =========================================================
        set_requires_grad(D, False)
        delta = G(mel_interp, mask)
        mel_fixed = mel_interp + mask * delta

        missing = mask.sum()
        loss_id = torch.tensor(0.0, device=device)

        if USE_IDENTITY_LOSS_FOR_K0 and (missing < 1.0):
            loss_mel = torch.tensor(0.0, device=device)
            loss_id  = delta.abs().mean()
        else:
            loss_mel = masked_l1(mel_fixed, mel_real, mask, 1.0, 0.0)

        # D forward for G loss
        d_fake_g, f_fake_g = D(pack_mel_for_D(mel_fixed, mask))
        with torch.no_grad():
            d_real_ng, f_real_ng = D(pack_mel_for_D(mel_real, mask))

        if MODE == "A2":
            loss_adv = g_adv_loss_lsgan(d_fake_g)
            loss_fm  = fm_loss(f_real_ng, f_fake_g)
        else:
            loss_adv = g_adv_loss_lsgan_ms(d_fake_g)
            loss_fm  = fm_loss_ms(f_real_ng, f_fake_g)

        adv_w = LAMBDA_ADV_MAX * min(1.0, step / max(1, ADV_WARMUP_STEPS))

        # discourage big edits inside the hole
        loss_delta = (mask * delta).abs().mean()

        if USE_STFT_LOSS:
            wav_hat = mel_to_wave_vocoder(mel_fixed)
            with torch.no_grad():
                wav_tgt = mel_to_wave_vocoder(mel_real)

            loss_stft = mrstft_loss(
                wav_hat, wav_tgt,
                fft_sizes=STFT_LOSS_FFTS,
                hop_sizes=STFT_LOSS_HOPS,
                win_sizes=STFT_LOSS_WINS,
                use_logmag=STFT_LOSS_LOGMAG,
            )
        else:
            loss_stft = torch.tensor(0.0, device=device)

        loss_g = (
            (LAMBDA_MEL * loss_mel) +
            (adv_w * loss_adv) +
            (LAMBDA_FM * loss_fm) +
            (LAMBDA_ID * loss_id) +
            (LAMBDA_DELTA * loss_delta) +
            (LAMBDA_STFT * loss_stft)
        )

        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_g.step()

        set_requires_grad(D, True)

        # discriminator score stats for logging
        with torch.no_grad():
            if MODE == "A2":
                dr_logit = d_real
                df_logit = d_fake
            else:
                dr_logit = d_real[0]  # first scale
                df_logit = d_fake[0]  # first scale

            dr_mean = float(dr_logit.mean().detach().cpu())
            df_mean = float(df_logit.mean().detach().cpu())
            dr_sig  = float(torch.sigmoid(dr_logit).mean().detach().cpu())
            df_sig  = float(torch.sigmoid(df_logit).mean().detach().cpu())

    # -------------------------
    # 4) logging
    # -------------------------
    if (step - start_step) % LOG_EVERY == 0:
        dt_min = (time.time() - t0) / 60.0
        with torch.no_grad():
            ref_mel = float(loss_mel.detach().cpu())
            base    = float(base_mel.detach().cpu())

            delta_mel = base - ref_mel
            ratio_mel = ref_mel / (base + 1e-8) if base > 0 else float("nan")
            win_mel   = 1.0 if ref_mel < base else 0.0

        print(
            f"step {step:7d} | lossD {float(loss_d):.3f} | lossG {float(loss_g):.3f} | "
            f"mel {ref_mel:.4f} adv {float(loss_adv):.3f} fm {float(loss_fm):.3f} "
            f"stft {float(loss_stft):.3f} id {float(loss_id):.4f} | "
            f"base {base:.4f} | ref {ref_mel:.4f} | "
            f"delta {delta_mel:+.4f} | ratio {ratio_mel:.3f} | win {win_mel:.0f} | "
            f"k={pair['k']} off={pair['offset']} | "
            f"Dmean real/fake {dr_mean:.3f}/{df_mean:.3f} "
            f"(sig {dr_sig:.3f}/{df_sig:.3f}) | "
            f"{dt_min:.1f} min"
        )

        with LOG_CSV.open("a", encoding="utf-8") as f:
            f.write(
                f"{step},{float(loss_d):.6f},{float(loss_g):.6f},"
                f"{float(loss_mel):.6f},{float(loss_adv):.6f},{float(loss_fm):.6f},{float(loss_stft):.6f},"
                f"{base:.6f},{ref_mel:.6f},{delta_mel:.6f},{ratio_mel:.6f},{win_mel:.0f},"
                f"{pair['k']},{pair['offset']},"
                f"{dr_mean:.6f},{df_mean:.6f},{dr_sig:.6f},{df_sig:.6f},{dt_min:.6f}\n"
            )

    # -------------------------
    # 5) checkpoint
    # -------------------------
    if (step - start_step) % CKPT_EVERY == 0:
        save_ckpt(step)

print("done")


 ## Quick eval on 1 speech + 1 music sample 

In [ ]:
@torch.no_grad()
def eval_one_file(path: Path, k=1, offset=0):
    w = safe_load_wav_mono(path, SR)[: int(SR * SEG_S)].to(device)
    wb = w.unsqueeze(0)

    mel_real = mel_spectrogram(
        wb,
        n_fft=h.n_fft, num_mels=h.num_mels, sampling_rate=h.sampling_rate,
        hop_size=h.hop_size, win_size=h.win_size, fmin=h.fmin, fmax=h.fmax,
        center=False,
    )
    mel_interp, mask = linear_time_inpaint(mel_real, k=k, offset=offset)
    delta = G(mel_interp, mask)
    mel_fixed = mel_interp + mask * delta

    base = masked_l1(mel_interp, mel_real, mask, 1.0, 0.0).item()
    ref  = masked_l1(mel_fixed,  mel_real, mask, 1.0, 0.0).item()

    delta_mel = base - ref
    ratio_mel = ref / (base + 1e-8) if base > 0 else float("nan")

    if USE_STFT_LOSS:
        wav_hat = mel_to_wave_vocoder(mel_fixed)
        wav_tgt = mel_to_wave_vocoder(mel_real)
        stft_val = float(mrstft_loss(
            wav_hat, wav_tgt,
            fft_sizes=STFT_LOSS_FFTS,
            hop_sizes=STFT_LOSS_HOPS,
            win_sizes=STFT_LOSS_WINS,
            use_logmag=STFT_LOSS_LOGMAG,
        ).item())
    else:
        stft_val = 0.0

    print(
        f"{path.name} | k={k} off={offset} | "
        f"base {base:.6f} | ref {ref:.6f} | "
        f"delta {delta_mel:+.6f} | ratio {ratio_mel:.3f} | stft {stft_val:.4f}"
    )
    return base, ref, delta_mel, ratio_mel, stft_val